# Player profiles V2

Set `season_variable` below, then use **Restart & Run All**. The notebook downloads published NBA Stats snapshots from [NBA Lineup Analyzer](https://github.com/meharpalbasi/NBA_Lineup202324_Analyzer/blob/main/docs/DATA_SOURCES.md), rather than calling stats.nba.com. Availability depends on the publisher providing all files for the chosen season. Data is regular season only. It preserves the original 50 games / 15 minutes thresholds, formulas, column names and percentile ranking. It reports missing data rather than inventing statistics. The output goes in the V2_output folder beside this notebook. Re-running replaces that season file in V2_output, while the original season CSVs remain untouched.

Install dependencies in your project environment: `pip install pandas nba_api notebook`.

For a catch-and-shoot or pull-up category with zero attempts and missing efficiency, V2 uses zero, matching the original notebook’s missing-category fallback. Those players are printed; a zero here is a convention, not a measured shooting percentage.

In [ ]:
season_variable = "2025-26"  # Change only this value to select the season.
games_threshold = 50
minutes_threshold = 15

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from nba_api.stats.static import players

if not re.fullmatch(r"20\d{2}-\d{2}", season_variable):
    raise ValueError("Season must look like 2025-26")
if (int(season_variable[:4]) + 1) % 100 != int(season_variable[-2:]):
    raise ValueError("Season end year must follow its start year")

base_url = "https://raw.githubusercontent.com/meharpalbasi/NBA_Lineup202324_Analyzer/main/data"
files = ["player_game_logs", "estimated_metrics", "shot_zones", "pt_shot_defender", "defense_tracking", "tracking"]
data = {}
for name in files:
    url = f"{base_url}/{name}_{season_variable}.csv"
    try:
        frame = pd.read_csv(url, low_memory=False)
    except Exception as exc:
        raise RuntimeError(f"Could not load {url}. The publisher may not provide this season or the network may be unavailable.") from exc
    if "SEASON_TYPE" not in frame.columns:
        raise ValueError(f"{name}: missing SEASON_TYPE")
    data[name] = frame.loc[frame.SEASON_TYPE.eq("Regular Season")].copy()
    if data[name].empty:
        raise ValueError(f"{name}: no regular season rows for {season_variable}")
    print(f"{name}: {len(data[name])} regular-season rows")

In [ ]:
# Qualify players using the same game-log counts and mean minutes as the original.
logs = data['player_game_logs'].copy()
if logs.duplicated(['PLAYER_ID', 'GAME_DATE']).any():
    raise ValueError('Duplicate player/date game logs found; inspect source before continuing.')
logs['MIN'] = pd.to_numeric(logs['MIN'], errors='raise')
summary_df = logs.groupby('PLAYER_ID').agg(GP_TOTAL=('MIN', 'size'), MIN_PG=('MIN', 'mean')).reset_index()
qualified_df = summary_df[(summary_df.GP_TOTAL >= games_threshold) & (summary_df.MIN_PG >= minutes_threshold)].copy()
qualified_df['MIN_PG'] = qualified_df['MIN_PG'].round(2)
if qualified_df.empty:
    raise ValueError('No qualified players found.')
lookup = {p['id']: p for p in players.get_players()}
missing_ids = sorted(set(qualified_df.PLAYER_ID) - set(lookup))
if missing_ids:
    raise ValueError(f'Player IDs missing from nba_api static list: {missing_ids}. Update nba_api.')
profiles = pd.DataFrame([lookup[int(pid)] for pid in qualified_df.PLAYER_ID])
df = qualified_df.merge(profiles, left_on='PLAYER_ID', right_on='id', validate='one_to_one')
print(f"Qualified players: {len(df)}")

In [ ]:
# Join each exact source by NBA player ID; reject missing or duplicate data.
def join_source(frame, key, fields, label):
    global df
    part = frame[[key] + fields].copy().rename(columns={key: 'PLAYER_ID'})
    if part.PLAYER_ID.duplicated().any():
        raise ValueError(f'{label}: duplicate player IDs')
    df = df.merge(part, on='PLAYER_ID', how='left', validate='one_to_one', indicator='_match')
    missing = df.loc[df['_match'] != 'both', ['PLAYER_ID', 'full_name']]
    if not missing.empty:
        for _, player in missing.iterrows():
            print(f"Missing source data: {player['full_name']} (ID {player['PLAYER_ID']}) — category: {label}")
        raise ValueError(f'{label}: required source rows are missing; no CSV was exported.')
    df = df.drop(columns='_match')

join_source(data['estimated_metrics'], 'PLAYER_ID', ['E_AST_RATIO','E_DREB_PCT','E_OREB_PCT','E_TOV_PCT'], 'estimated metrics')
join_source(data['shot_zones'], 'PLAYER_ID', ['RA_FGM','RA_FGA','PAINT_FGM','PAINT_FGA','MID_FGA','MID_FG_PCT','C3_FGA','C3_FG_PCT','ATB3_FGA','ATB3_FG_PCT'], 'shot zones')
for category in ['VERY_TIGHT', 'TIGHT', 'OPEN', 'WIDE_OPEN']:
    source = data['pt_shot_defender']
    source = source[source.CLOSE_DEF.eq(category)][['PLAYER_ID', 'FGA', 'FG2M', 'FG3M']].rename(columns={x: f'{category}_{x}' for x in ['FGA','FG2M','FG3M']})
    join_source(source, 'PLAYER_ID', [f'{category}_{x}' for x in ['FGA','FG2M','FG3M']], category)
for category, prefix, fga in [('Less Than 10Ft','paint_def','FGA_LT_10'), ('Greater Than 15Ft','perimeter_def','FGA_GT_15')]:
    source = data['defense_tracking']
    source = source[source.DEFENSE_CATEGORY.eq(category)][['CLOSE_DEF_PERSON_ID', fga, 'PLUSMINUS']].rename(columns={fga: prefix+'_fga', 'PLUSMINUS': prefix+'_plusminus'})
    join_source(source, 'CLOSE_DEF_PERSON_ID', [prefix+'_fga', prefix+'_plusminus'], category)
for category, prefix in [('CatchShoot', 'CATCH_SHOOT'), ('PullUpShot', 'PULL_UP')]:
    source = data['tracking']
    source = source[(source.PT_MEASURE_TYPE == category) & (source.PLAYER_OR_TEAM == 'Player')]
    join_source(source, 'PLAYER_ID', [prefix+'_FGA', prefix+'_EFG_PCT'], category)
print('All required player rows matched.')

In [ ]:
# Recreate the original raw metrics; volumes use the original rounded minutes per game.
minutes = df.GP_TOTAL * df.MIN_PG
raw = pd.DataFrame(index=df.index)
for target, source in [('AST_PCT','E_AST_RATIO'), ('DREB_PCT','E_DREB_PCT'), ('OREB_PCT','E_OREB_PCT'), ('TOV_PCT','E_TOV_PCT')]:
    raw[target] = df[source]
raw['Midrange_volume_pm'] = df.MID_FGA / minutes
raw['Midrange_efficiency'] = df.MID_FG_PCT
raw['Paint_volume_pm'] = (df.PAINT_FGA + df.RA_FGA) / minutes
raw['Paint_efficiency'] = (df.PAINT_FGM + df.RA_FGM) / (df.PAINT_FGA + df.RA_FGA).replace(0, np.nan)
raw['Corner_volume_pm'] = df.C3_FGA / minutes
raw['Corner_efficiency'] = df.C3_FG_PCT
raw['Above_the_break_volume_pm'] = df.ATB3_FGA / minutes
raw['Above_the_break_efficiency'] = df.ATB3_FG_PCT
for name, a, b in [('Tight', 'VERY_TIGHT', 'TIGHT'), ('Open', 'WIDE_OPEN', 'OPEN')]:
    attempts = df[a+'_FGA'] + df[b+'_FGA']
    raw[name+'_volume_pm'] = attempts / minutes
    raw[name+'_efficiency'] = (df[a+'_FG2M'] + df[b+'_FG2M'] + 1.5 * (df[a+'_FG3M'] + df[b+'_FG3M'])) / attempts.replace(0, np.nan)
for name, prefix in [('Catch_and_Shoot', 'CATCH_SHOOT'), ('Pull_up', 'PULL_UP')]:
    raw[name+'_Volume'] = df[prefix+'_FGA'] / minutes
    # The original notebook uses zero when a shooting category is absent.
    # The published tracking file reports no efficiency when attempts are zero.
    no_attempts = (df[prefix+'_FGA'] == 0) & df[prefix+'_EFG_PCT'].isna()
    if no_attempts.any():
        print(f"{name}: using 0 for zero attempts: {df.loc[no_attempts, 'full_name'].tolist()}")
    raw[name+'_Efficiency'] = df[prefix+'_EFG_PCT'].mask(no_attempts, 0.0)
for name, prefix in [('Paint', 'paint_def'), ('Perimeter', 'perimeter_def')]:
    raw[name+'_defense_volume_pm'] = df[prefix+'_fga'] / minutes
    raw[name+'_defense_deterance'] = df[prefix+'_plusminus']
# Treat non-numeric, missing and infinite values as errors, while allowing real zeros.
raw = raw.apply(pd.to_numeric, errors='coerce')
invalid = ~np.isfinite(raw.to_numpy())
if invalid.any():
    for row_index, column_index in np.argwhere(invalid):
        print(f"Missing or undefined: {df.iloc[row_index]['full_name']} "
              f"(ID {df.iloc[row_index]['PLAYER_ID']}) — category: {raw.columns[column_index]}")
    raise ValueError('Required values are missing or undefined; no CSV was exported.')
print('All raw metrics are present and finite.')

In [ ]:
# Rank within the qualified players, matching the original column order and schema.
identity_columns = ['id', 'full_name', 'first_name', 'last_name', 'is_active', 'GP_TOTAL', 'MIN_PG']
df_percentile = df[identity_columns].copy()
for column in raw.columns:
    descending = column in ['Paint_defense_deterance', 'Perimeter_defense_deterance']
    df_percentile[column+'_percentile'] = raw[column].rank(pct=True, ascending=not descending)

output_dir = Path.cwd()
output_dir.mkdir(exist_ok=True)
output = output_dir / f'{season_variable}.csv'
df_percentile.to_csv(output, index=False)
print(f'Saved {len(df_percentile)} players and {len(df_percentile.columns)} columns to {output}')
display(df_percentile.head())